# 2110446 DATA SCIENCE AND DATA ENGINEERING

## **Midterm:** Traffy Fondue

- **Author:** Worralop Srichainont
- **Year:** 2025 (Semester 2)

## **Notebook I:** Data Preprocessing

In this notebook, we are focus on cleaning the data before training the model.

# **Part 0:** Dependencies

In [1]:
!pip -q install pythainlp python-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.2 MB/s eta 0:00:00


In [2]:
import re
import string
import pandas as pd

from tqdm import tqdm
from pythainlp.util import normalize
from pythainlp.phayathaibert import ThaiTextProcessor

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.26M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.4M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

# **Part 1:** Load Dataset

In [3]:
TRAIN_FILE_PATH = "/kaggle/input/datasets/worralopsrichainont/2110446-dsde-midterm-dataset-cleaned/train.csv"
TEST_FILE_PATH = "/kaggle/input/datasets/worralopsrichainont/2110446-dsde-midterm-dataset-cleaned/test.csv"

In [4]:
df_train = pd.read_csv(TRAIN_FILE_PATH)
df_test = pd.read_csv(TEST_FILE_PATH)

Get brief details of the dataset

In [5]:
print(f"The train dataset has {df_train.shape[0]} rows and {df_train.shape[1]} columns")
for col_name in df_train.columns:
    print(f"- {col_name}")

The train dataset has 306419 rows and 14 columns
- id
- comment
- สำนักงานตำรวจแห่งชาติ
- การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย
- สภาเด็กและเยาวชนกรุงเทพมหานคร
- กรมควบคุมมลพิษ
- กรมสรรพสามิต
- การไฟฟ้านครหลวง
- กรมทางหลวง
- สำนักงานประกันสุขภาพแห่งชาติ
- การประปานครหลวง
- คณะกรรมการการพัฒนาเศรษฐกิจ
- กระทรวงการท่องเที่ยวและกีฬา
- สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200


In [6]:
print(f"The test dataset has {df_test.shape[0]} rows and {df_test.shape[1]} columns")
for col_name in df_test.columns:
    print(f"- {col_name}")

The test dataset has 37406 rows and 2 columns
- id
- comment


Display `DataFrame`

In [7]:
df_train.head()

,id,comment,สำนักงานตำรวจแห่งชาติ,การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย,สภาเด็กและเยาวชนกรุงเทพมหานคร,กรมควบคุมมลพิษ,กรมสรรพสามิต,การไฟฟ้านครหลวง,กรมทางหลวง,สำนักงานประกันสุขภาพแห่งชาติ,การประปานครหลวง,คณะกรรมการการพัฒนาเศรษฐกิจ,กระทรวงการท่องเที่ยวและกีฬา,สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200
0,0,ทำไมปล่อยให้จุดพลุกันสนั่นหวั่นไหว,0,0,0,0,0,0,0,0,0,0,0,0
1,1,แจ้งว่าการจุดพลุต้องขออนุญาต ทำไมจุดกันมากมายข...,0,0,0,0,0,0,0,0,0,0,0,0
2,2,คาดว่ามีการจุดพลุไม่ขอทางกรุงเทพให้ถูกต้อง ส่ง...,0,0,0,0,0,0,0,0,0,0,0,0
3,3,ไม่แน่ใจ กทม อนุญาตให้ร้านชอคโกแลตวิลจุพลุถึงก...,0,0,0,0,0,0,0,0,0,0,0,0
4,4,ไม่ทราบใครจัดงานปีใหม่ละแวกนี้ เปิดเสียงเพลงดั...,0,0,0,0,0,0,0,0,0,0,0,0


In [8]:
df_test.head()

,id,comment
0,0,รถติดจังเลยครับ อยากได้เกาะกลาง ที่ขยับเพิ่มเล...
1,1,ในซอยมีการเตรียมทำท่อระบายน้ำ โดยผู้รับเหมา มา...
2,2,มีต้นไม้กีดขวางทางสัญจรไปมาทำให้เกิดอันตราย
3,3,ร้านนวดบริเวณนี้วางของเกะกะบนทางเท้ามากมาย
4,4,ศูนย์เรื่องราวร้องทุกข์ ได้รับการประสานผ่านระบ...


# **Part 2:** Data Preparation

## 2.1. Utility Class

In [9]:
HTML_PATTERN = re.compile(r"<[^<>]*>")

PHONE_PATTERN = re.compile(
    r"(?<![\d/.:])(?:"
    r"0[689]\d[- ]?\d{3}[- ]?\d{4}|"
    r"02[- ]?\d{3}[- ]?\d{4}|"
    r"0[3457]\d[- ]?\d{3}[- ]?\d{3}|"
    r"(?:1800|1900|1401)[- ]?\d{3}[- ]?\d{3}|"
    r"1555|1129|1125|1137|1644|1669|191|199"
    r")(?![\d/.:])"
)

EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")

URL_PATTERN = re.compile(r"https?://[^\sก-๛]+|www\.[^\sก-๛]+")

ZERO_WIDTH_CHARS_PATTERN = re.compile(r"[\u200B\u200C\u200D\uFEFF]+")

PUNCTUATION_PATTERN = re.compile(r'([!"\'(),./:;?#-])\1+')

ALLOWED_PUNC = r"""!"'(),./:;?#-"""
WHITELIST_PATTERN = re.compile(rf"[^a-zA-Z0-9ก-๛\s{re.escape(ALLOWED_PUNC)}฿]")

WHITESPACE_PATTERN = re.compile(r"\s+")

In [10]:
class TextCleanerBERT:
    THAI_TEXT_PROCESSOR = ThaiTextProcessor()

    @staticmethod
    def get_cleaned_thai_text(text):
        if not isinstance(text, str):
            return text

        text = text.lower().strip()
        text = TextCleanerBERT.remove_html(text)
        text = TextCleanerBERT.remove_telephone_number(text)
        text = TextCleanerBERT.remove_email(text)
        text = TextCleanerBERT.remove_url(text)
        text = TextCleanerBERT.remove_zero_width_chars(text)
        text = TextCleanerBERT.remove_non_whitelist_chars(text)
        text = TextCleanerBERT.clean_punctuation(text)
        text = TextCleanerBERT.pad_hashtags(text)

        text = normalize(text)
        text = TextCleanerBERT.remove_redundant_chars(text)
        text = TextCleanerBERT.clean_whitespace(text)
        return text

    @staticmethod
    def remove_html(text):
        return HTML_PATTERN.sub("", text)

    @staticmethod
    def remove_telephone_number(text):
        return PHONE_PATTERN.sub("", text)

    @staticmethod
    def remove_email(text):
        return EMAIL_PATTERN.sub("", text)

    @staticmethod
    def remove_url(text):
        return URL_PATTERN.sub("", text)

    @staticmethod
    def remove_zero_width_chars(text):
        return ZERO_WIDTH_CHARS_PATTERN.sub("", text)

    @staticmethod
    def remove_non_whitelist_chars(text):
        return WHITELIST_PATTERN.sub(" ", text)

    @staticmethod
    def clean_punctuation(text):
        return PUNCTUATION_PATTERN.sub(r"\1", text)

    @staticmethod
    def pad_hashtags(text):
        return text.replace("#", " # ")

    @staticmethod
    def remove_redundant_chars(text):
        return TextCleanerBERT.THAI_TEXT_PROCESSOR.replace_rep_after(text)

    @staticmethod
    def clean_whitespace(text):
        text = WHITESPACE_PATTERN.sub(" ", text)
        return text.strip()

## 2.2. Data Cleaning

Remove missing value from the train dataset

In [11]:
df_train = df_train.dropna()

Apply text cleaning utility function on the text

In [12]:
tqdm.pandas(desc="Cleaning Train Dataset")
df_train["comment"] = df_train["comment"].progress_apply(
    TextCleanerBERT.get_cleaned_thai_text
)

Cleaning Train Dataset: 100%|██████████| 304284/304284 [00:59<00:00, 5077.50it/s]


In [13]:
tqdm.pandas(desc="Cleaning Test Dataset")
df_test["comment"] = df_test["comment"].progress_apply(
    TextCleanerBERT.get_cleaned_thai_text
)

Cleaning Test Dataset: 100%|██████████| 37406/37406 [00:07<00:00, 4858.61it/s]


## 2.3. Display Cleaned Dataset

In [14]:
df_train.head()

,id,comment,สำนักงานตำรวจแห่งชาติ,การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย,สภาเด็กและเยาวชนกรุงเทพมหานคร,กรมควบคุมมลพิษ,กรมสรรพสามิต,การไฟฟ้านครหลวง,กรมทางหลวง,สำนักงานประกันสุขภาพแห่งชาติ,การประปานครหลวง,คณะกรรมการการพัฒนาเศรษฐกิจ,กระทรวงการท่องเที่ยวและกีฬา,สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200
0,0,ทำไมปล่อยให้จุดพลุกันสนั่นหวั่นไหว,0,0,0,0,0,0,0,0,0,0,0,0
1,1,แจ้งว่าการจุดพลุต้องขออนุญาต ทำไมจุดกันมากมายข...,0,0,0,0,0,0,0,0,0,0,0,0
2,2,คาดว่ามีการจุดพลุไม่ขอทางกรุงเทพให้ถูกต้อง ส่ง...,0,0,0,0,0,0,0,0,0,0,0,0
3,3,ไม่แน่ใจ กทม อนุญาตให้ร้านชอคโกแลตวิลจุพลุถึงก...,0,0,0,0,0,0,0,0,0,0,0,0
4,4,ไม่ทราบใครจัดงานปีใหม่ละแวกนี้ เปิดเสียงเพลงดั...,0,0,0,0,0,0,0,0,0,0,0,0


In [15]:
df_test.head()

,id,comment
0,0,รถติดจังเลยครับ อยากได้เกาะกลาง ที่ขยับเพิ่มเล...
1,1,ในซอยมีการเตรียมทำท่อระบายน้ำ โดยผู้รับเหมา มา...
2,2,มีต้นไม้กีดขวางทางสัญจรไปมาทำให้เกิดอันตราย
3,3,ร้านนวดบริเวณนี้วางของเกะกะบนทางเท้ามากมาย
4,4,ศูนย์เรื่องราวร้องทุกข์ ได้รับการประสานผ่านระบ...


# **Part 3:** Save as CSV

In [16]:
df_train.to_csv("train_cleaned.csv", index=False, encoding="utf-8-sig")

In [17]:
df_test.to_csv("test_cleaned.csv", index=False, encoding="utf-8-sig")